In [1]:
from dotenv import load_dotenv 
load_dotenv()

True

In [2]:
import os 
from getpass import getpass 

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [4]:
#Quick illustration of the two approaches deterministic and model based

import re 

#----- Deterministic approach -----
def deterministic_guardrail(text:str) -> bool:
    """ Returns True if content is blocked. """
    banned_keywords = ["hack", "exploit", "malware", "bomb"]

    return any( kw in text.lower() for kw in banned_keywords )

test_inputs = [
    "How do i hack into a database?",
    "What is the captial of France",
    "Explain how malware spreads",
]

print("=== Deterministic Guradrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "BLOCKED" if blocked else "Allowed"
    print(f"{status}: {inp}")


=== Deterministic Guradrail Demo ===
BLOCKED: How do i hack into a database?
Allowed: What is the captial of France
BLOCKED: Explain how malware spreads


In [5]:
from langchain_openai import ChatOpenAI 

# -- Model based approach --- 
def model_based_guardrail(text:str) -> str:
    """ Uses an LLM to evaluate content saftey. Returns SAFE or UNSAFE."""
    model = ChatOpenAI(model = "gpt-4o-mini", temperature=0)
    prompt = f"""Is the following user input safe to process?
    Reply with only 'SAFE' or 'UNSAFE'.

    Input: {text}"""
    result = model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

print("===Model Based Guardrails Demo ===")
for inp in test_inputs:
     verdict = model_based_guardrail(inp)
     status = "UNSAFE " if "UNSAFE" in verdict else "SAFE"
     print(f"{status}: {inp}")

===Model Based Guardrails Demo ===
UNSAFE : How do i hack into a database?
SAFE: What is the captial of France
SAFE: Explain how malware spreads


In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI 
from langchain_core.tools import tool 

#Define a simple dummy tool 
@tool 
def customer_lookup(query:str) -> str:
    """Look up customer information"""
    return f"Customer record found for query: {query}"

#Create agent with PII Middleware 
agent = create_agent(
    model = "gpt-4o",
    tools = [customer_lookup],
    middleware = [
        PIIMiddleware(
            "email",
            strategy = "redact",
            apply_to_input=True,
        ),

        #Mask credit cards in user input 
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        #Block API Keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector = r"sk-[a-zA-Z0-9]{32}",
            strategy = "block",
            apply_to_input = True,
        ),
    ],
)

print("Agent with PII moddleware created successfully!")

Agent with PII moddleware created successfully!


In [8]:
#Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role":"user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I found customer records for both the email address and the card number you provided. How can I assist you further with your account?


In [9]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='aaa86bf2-941b-4a4a-83ca-82c9181bb6a5'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 70, 'total_tokens': 127, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_40aced2ba4', 'id': 'chatcmpl-EPdLT1zcZrfFafWGqNa4Z5IjNIJnq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0b717-41b8-77b3-86d5-05c8f57ad5f5-0', tool_calls=[{'name': 'customer_lookup', 'args': {'q